In [387]:
import numpy as np
from colossus.cosmology import cosmology
cosmology.setCosmology('planck15') 
from colossus.lss import mass_function as mf 
import glob
import numexpr as ne
import sys
from scipy.optimize import newton
%matplotlib widget


files = [f.split('a')[1].split('.d')[0] for f in glob.glob('ssfrs/ssfr_a*.dat')]
a_list = np.array([float(a) for a in files])
mass_list = np.array([np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,0] for f in files])
ssfr_list = [np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,1] for f in files]
    
param_file = np.loadtxt("smhm_params.txt")
names = "EFF_0 EFF_0_A EFF_0_A2 EFF_0_Z M_1 M_1_A M_1_A2 M_1_Z ALPHA ALPHA_A ALPHA_A2 ALPHA_Z BETA BETA_A BETA_Z DELTA GAMMA GAMMA_A GAMMA_Z CHI2".split(" ");
params = dict(zip(names, param_file[:,1]))

def create_ranges_numexpr(start, stop, N):

    divisor = N-1
    s0 = start[:,None]
    s1 = stop[:,None]
    r = np.arange(N)

    return ne.evaluate('((1.0/divisor) * (s1 - s0))*r + s0')

def grab_obs(redshift):
    
    obs_points = []
    with open("obs_collect.dat", 'r') as f:
        for line in f:
            if line[0:4] != ';;//':
                obs_points.append(line.split())
    x = []
    y = []
    yerr = []
    for i in obs_points:
        if float(i[0]) == redshift:
            x.append(float(i[1]))
            y.append(float(i[2]))
            yerr.append(float(i[3]))
            
    return(x,y,yerr)



class QLF():
    def __init__(self, z, bin_num):
        
        
        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bin_num = bin_num
        self.get_zparams()
        
        self.max_halo = 15.
        self.HaloBins = np.linspace(7., self.max_halo, bin_num)
        slopes = self.get_slope(self.HaloBins)
        while slopes[-1] < 0:
            self.max_halo -= .1
            self.HaloBins = np.linspace(7., self.max_halo, bin_num)
            slopes = self.get_slope(self.HaloBins)
            
        self.max_stell = self.get_Mstar(self.max_halo)
        
        self.fp = self.HaloBins
        self.xp = self.get_Mstar(self.fp)
        
        self.LumBins = np.linspace(5., 16., bin_num)
        self.StellBins = np.linspace(5.,self.max_stell, bin_num)
        
        
    def get_zparams(self):
        a1 = self.a - 1.0
        lna = np.log(self.a)
        self.zparams = {}
        self.zparams['m_1'] = params['M_1'] + a1*params['M_1_A'] - lna*params['M_1_A2'] + self.z*params['M_1_Z']
        self.zparams['sm_0'] = self.zparams['m_1'] + params['EFF_0'] + a1*params['EFF_0_A'] - lna*params['EFF_0_A2'] + self.z*params['EFF_0_Z']
        self.zparams['alpha'] = params['ALPHA'] + a1*params['ALPHA_A'] - lna*params['ALPHA_A2'] + self.z*params['ALPHA_Z']
        self.zparams['beta'] = params['BETA'] + a1*params['BETA_A'] + self.z*params['BETA_Z']
        self.zparams['delta'] = params['DELTA']
        self.zparams['gamma'] = 10**(params['GAMMA'] + a1*params['GAMMA_A'] + self.z*params['GAMMA_Z'])
        
    
    def get_slope(self, Mhalo):

        dm = Mhalo-self.zparams['m_1'];
        term1 = (self.zparams['alpha']*10.**(self.zparams['beta']*dm)+self.zparams['beta']*10.**(self.zparams['alpha']*dm))/(10.**(self.zparams['beta']*dm) + 10.**(self.zparams['alpha']*dm))
        term2 = -self.zparams['gamma']*dm*np.exp(-(dm/self.zparams['delta'])**2/2.)/self.zparams['delta']**2
        slope = term1 + term2

        return slope
    
    def get_Mstar(self, Mhalo):
    
        dm = Mhalo-self.zparams['m_1']
        dm2 = dm/self.zparams['delta']
        Mstar = self.zparams['sm_0'] - np.log10(10**(-self.zparams['alpha']*dm) + 10**(-self.zparams['beta']*dm)) + self.zparams['gamma']*np.exp(-0.5*(dm2*dm2))

        return Mstar
    
    
    def get_Mhalo(self, Mstar):
        
        Mhalo = np.interp(Mstar, self.xp, self.fp)
        
        return Mhalo
    
    
    def get_SMBM(self, dM, Mmid, slope1 = 0.2, slope3 = 1.):

        start = [7., np.log10(1.4*10**4.)]
        stop = [12., np.log10(1.4*10**9.)]
        mstar1 = Mmid - dM
        mstar2 = Mmid + dM
        int1 = start[1] - start[0] * slope1
        int3 = stop[1] - stop[0] * slope3
        x = (int3 - int1) / (slope1 - slope3)
        y = slope1 * x + int1
        if mstar1 < x:
            mstar1 = x
        mbh1 = slope1 * mstar1 + int1
        mbh2 = mstar2 + int3
        slope2 = (mbh2 - mbh1) / (mstar2 - mstar1)
        int2 = mbh2 - mstar2 * slope2

        self.slope_list, self.int_list, self.mass_cuts = [slope1, slope2, slope3], [int1, int2, int3], [mstar1, mstar2] 
    
    
    def gauss_array(self, vals, std, amp):

        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(vals[-1]-vals[:-1])**2.0)/(2.0*std**2))

        return y

    
    def convolve_smhm(self, StellBins, sigma, bin_num, z): 

        halomasses = self.get_Mhalo(np.asarray(StellBins))
        plus_mins = (5.0 * sigma) / self.get_slope(np.asarray(halomasses))
        mins = halomasses - plus_mins
        maxs = halomasses + plus_mins
        mins[mins < 7.] = 7.
        maxs[maxs > self.max_halo] = self.max_halo
        MHalo = create_ranges_numexpr(mins, maxs, bin_num)
        dNdMhalo = mf.massFunction(10.**MHalo, z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16') * np.log(10)
        meanMstar = np.apply_along_axis(self.get_Mstar, 1, MHalo)
        values = np.zeros((bin_num,bin_num+1))
        values[:,-1] = StellBins
        values[:,:-1] = meanMstar
        Mstar_prob = np.apply_along_axis(self.gauss_array, 1, values, sigma, 1)
        dNdMstar = np.sum(Mstar_prob * dNdMhalo, axis = 1) * (MHalo[:,1] - MHalo[:,0])

        return dNdMstar
    
    
    
    def get_dNdMstar(self, smhm_scat):
        
        if smhm_scat == 0.:
            self.dNdMstar = mf.massFunction(10.**self.get_Mhalo(self.StellBins), self.z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')  / (get_slope(um.get_Mhalo(self.StellBins, self.z)) * np.log10(np.e))
        else:
            self.dNdMstar = self.convolve_smhm(self.StellBins, smhm_scat, self.bin_num, self.z)


            
    def get_dNdMbh(self):
        
        self.early = (self.StellBins <= self.mass_cuts[0])
        self.growth = ((self.StellBins > self.mass_cuts[0]) & (self.StellBins < self.mass_cuts[1]))
        self.late = (self.StellBins > self.mass_cuts[1])
        self.m = np.zeros(len(self.StellBins))
        self.m[self.early] = self.slope_list[0]
        self.m[self.growth] = self.slope_list[1]
        self.m[self.late] = self.slope_list[2]
        self.dNdMbh = self.dNdMstar / self.m
        
        
    def etas(self, Mbh):
    
        n = np.asarray(self.LumBins) - np.log10(3.3e4) - Mbh

        return n
        
        
        
            
    def get_mean_etas(self, vals, a, xsigs, files = files):

        Mbh = vals[0]
        Mstar = vals[1]
        slope = vals[2]
        if slope == self.slope_list[0]:
            xsig = xsigs[0]
        else:
            xsig = xsigs[1]
        closest_a = np.argmin(np.abs(a_list - a))
        closest_m = np.argmin(np.abs(mass_list[closest_a] - Mstar))
        ssfr = ssfr_list[closest_a][closest_m]

        Ledd = 1.3*10**31 * 10**Mbh #J/s 
        Mdotedd = Ledd / (.1 * (2.99*10**8)**2) #kg/s???
        sbhr = slope*(ssfr/(3.154*10**7)) #1/s??
        Mdotbh = sbhr*(10**Mbh*2*10**30)
        eta = Mdotbh/Mdotedd
        nsig = xsig * np.log10(Mdotbh)/np.log10(Mdotedd)
        
        return np.log10(eta), nsig
    
#     def get_mean_etas(self, vals, a, xsigs, files = files):

#         Mbh = vals[0]
#         Mstar = vals[1]
#         slope = vals[2]
#         if slope == self.slope_list[0]:
#             xsig = xsigs[0]
#         else:
#             xsig = xsigs[1]
#         closest_a = np.argmin(np.abs(a_list - a))
#         closest_m = np.argmin(np.abs(mass_list[closest_a] - Mstar))
#         ssfr = ssfr_list[closest_a][closest_m]

#         Ledd = 1.3*10**31 * 10**Mbh #J/s 
#         Mdotedd = Ledd / (.1 * (2.99*10**8)**2) #kg/s???
#         sbhr = slope*(ssfr/(3.154*10**7)) #1/s??
#         Mdotbh = sbhr*(10**Mbh*2*10**30)
#         eta = Mdotbh/Mdotedd
#         nsig = xsig * np.log10(eta)
        
#         return np.log10(Mdotbh/(2*10**30*3.17098e-8)), xsig
    
    
    def gauss(self, x, *var):
  
        mean, std, amp = var
        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(x-mean)**2.0)/(2.0*std**2))

        return y
    
    
    def prob_eddratios(self, vals):

        probdens = self.gauss(vals[:-2], vals[-2]-(vals[-1]**2)/2, vals[-1], 1)

        return probdens
    
    def get_dNdL(self, xsigs, obscured):
        
        ###exp sig growth
            
        b = np.zeros(self.bin_num)
        b[self.early] = self.int_list[0]
        b[self.growth] = self.int_list[1]
        b[self.late] = self.int_list[2]

        leftc = np.argmin(self.early)
        rightc = np.argmax(self.late)
        per = len(self.growth[self.growth==True])*.1
        cut1l = int(leftc - per)
        cut1r = int(leftc + per + 1)
        cut2l = int(rightc - per)
        cut2r = int(rightc + per + 1)

        BHBins = self.StellBins * self.m + b
        eta_lists = np.apply_along_axis(self.etas, 1, np.reshape(BHBins,(self.bin_num,1)))

        vals = np.zeros((self.bin_num,3))
        vals[:,0] = BHBins
        vals[:,1] = self.StellBins
        vals[:,2] = self.m
        mean_etas = np.apply_along_axis(self.get_mean_etas, 1, vals, self.a, xsigs)
            
        vals = np.zeros((self.bin_num, self.bin_num+2))
        vals[:,:-2] = eta_lists
        vals[:,-2] = mean_etas[:,0]
        vals[:,-1] = mean_etas[:,1]
        
        
        self.plotpurp = mean_etas
        self.eddratios = np.apply_along_axis(self.prob_eddratios, 1, vals)
        self.eddratiobins = eta_lists
        
        self.dNdL = np.log10((1-obscured) * (np.sum(np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1)), axis = 0)))
        ind_dNdL_off = (1-obscured) * np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1))
        self.ind_dNdL = np.zeros((self.bin_num,self.bin_num))
        c = 0
        for l in ind_dNdL_off:
            self.ind_dNdL[c,:] = l
            c += 1
        

In [413]:
plt.close('all')
import matplotlib.pyplot as plt
bins = 500
fig = plt.figure(figsize=(5,5))
z=3
obscured = .8
sig_X = [1.25,.85]
dM = 0.55
mMid = 10.3
smhm_scat = 0.3
qlf = QLF(z, bins)
qlf.get_dNdMstar(smhm_scat)
qlf.get_SMBM(dM, mMid)
qlf.get_dNdMbh()
qlf.get_dNdL(sig_X, obscured)

# for z, col in zip([0.0,1.0,2.0,3.0,4.0,5.0,6.0],['red','orange','green','blue','violet','indigo']):
#     obscured = .75
#     qlf = QLF(z, bins)
#     qlf.get_dNdMstar(0.3)
#     qlf.get_SMBM(.45, 10.3)
#     qlf.get_dNdMbh()
#     qlf.get_dNdL([.8,.4], obscured)
#     xm, ym = qlf.LumBins, qlf.dNdL
    
#     x = np.linspace(-16,-5,200)
#     count = 0
#     for mu, sig, i in zip(qlf.plotpurp[:,0],qlf.plotpurp[:,1],qlf.m):
#         if count == 0:
#             plt.plot(x,qlf.gauss(x,mu,sig,1), c=col,linewidth=0.75, linestyle = ls, label = str(z))
#             count+=1
#         else:
#             plt.plot(x,qlf.gauss(x,mu,sig,1), c=col,linewidth=0.5, linestyle = ls)
#     for mu in qlf.plotpurp[:,0]:
#         plt.plot(z,mu)

# plt.xlabel(r'$log_{10} [\dot{M}_{BH}/M_{\odot}] s^{-1}$')
# plt.legend()

# xm, ym = qlf.LumBins, qlf.dNdL
# plt.plot(xm,ym,c='k')
# ym_in = qlf.ind_dNdL
# col = np.zeros(qlf.bin_num)
# col[qlf.early] = 0
# col[qlf.growth] = 1
# col[qlf.late] = 2
# for l, c in zip(ym_in, col):
#     c_list = ['r','gray','gray']
#     plt.plot(xm, np.log10(l), linewidth = .5, color = c_list[int(c)])
# plt.axis([5.8,16,-10,0])
# x, y , yerr = grab_obs(z)
# plt.errorbar(x, y, yerr = yerr, fmt = 'o', markersize = .5,color='blue')
# plt.text(10,-1,'z = '+str(z))
# plt.text(10,-1.75,'dM = '+str(dM))
# plt.text()
# plt.text(14,-2.5,r'$\sigma_{X} = $'+str(sig_X[0]),color='r')
# plt.text(14,-1.75,r'$\sigma_{X} = $'+str(sig_X[1]),color='gray')
# plt.plot()

x = np.linspace(-8,8,500)
for mean, sig in zip(qlf.plotpurp[:,0], qlf.plotpurp[:,1]):
    y = qlf.gauss(x, mean, sig, 1)
    plt.plot(x,y)
print(qlf.plotpurp[:,1])

FigureCanvasNbAgg()

[1.13420743 1.13422336 1.13423929 1.13425521 1.13427113 1.13428704
 1.13430295 1.13431885 1.13433476 1.13435065 1.13436654 1.13438243
 1.13439831 1.13441419 1.13443007 1.13444593 1.1344618  1.13447766
 1.13449352 1.13450937 1.13452522 1.13454106 1.1345569  1.13457274
 1.13458857 1.13460439 1.13462021 1.13463603 1.13465185 1.13466765
 1.13468346 1.13469926 1.13471506 1.13473085 1.13474663 1.13476242
 1.1347782  1.13479397 1.13480974 1.13482551 1.13484127 1.13485702
 1.13487278 1.13488853 1.13490427 1.13492001 1.13493574 1.13495148
 1.1349672  1.13498293 1.13499864 1.13501436 1.13503007 1.13504577
 1.13506147 1.13507717 1.13509286 1.13510855 1.13512424 1.13513992
 1.13515559 1.13517126 1.13518693 1.13520259 1.13521825 1.1352339
 1.13524955 1.1352652  1.13528084 1.13529648 1.13531211 1.13532774
 1.13534336 1.13535898 1.1353746  1.13539021 1.13540582 1.13542142
 1.13543702 1.13545261 1.1354682  1.13548379 1.13549937 1.13551495
 1.13553052 1.13554609 1.13556165 1.13557722 1.13559277 1.13560

$    L_{Edd} = const \frac{M_{BH}}{M_{\odot}} = \epsilon \dot{M}_{Edd} c^2    $

$    \frac{dM_{BH}}{dt} = \frac{M_{BH}}{M_{*}} \frac{dlogM_{BH}}{dlogM_{*}} \frac{dM_{*}}{dt}    $

$    \eta = \frac{\dot{M}_{BH}}{\dot{M}_{Edd}}    $

$    X = \frac{\dot{M}_{BH}}{<\dot{M}_{BH}>}    $

$    <\dot{M}_{BH}> = M_{BH} \frac{dlogM_{BH}}{dlogM_{*}} \frac{<\dot{M}_*>}{M_*}    $

$    <\eta> = \frac{<\dot{M}_{BH}>}{\dot{M}_{Edd}} = \frac{M_{BH}}{\dot{M}_{Edd}} \frac{dlogM_{BH}}{dlogM_{*}} \frac{<\dot{M}_*>}{M_*}      $

$    \sigma_{\eta} = \sigma_{X} log \big[ \frac{<\dot{M}_{BH}>}{\dot{M}_{Edd}} \big] = \sigma_{X} log\eta    $

In [240]:
def gauss(x, var):

    mean, std, amp = var
    y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(x-mean)**2.0)/(2.0*std**2))

    return y

params = [0, np.sqrt(.2), 1]
x = np.linspace(-5,5,200)
y = gauss(x,params)
plt.plot(x,y)

params = [0, 1, 1]
x = np.linspace(-5,5,200)
y = gauss(x,params)
plt.plot(x,y)

params = [0, np.sqrt(5.), 1]
x = np.linspace(-5,5,200)
y = gauss(x,params)
plt.plot(x,y)


xn = x/2
plt.plot(xn,y)
print(np.var(x), np.var(xn))


8.41708542713568 2.10427135678392


Code to create universal $<\dot{M}_{BH}>$, z evolution plot.

In [247]:
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib
plt.close('all')
bins = 200
fig, ax = plt.subplots(figsize=(6,6))
cbax = fig.add_axes([0.05, 0.35, .9, 0.05]) 
fig.subplots_adjust(bottom=0.5)
dM = .5
obscured = .75
zlist = np.linspace(0,6,500)
mulist = []
cut = []
for z in zlist:
    obscured = .75
    qlf = QLF(z, bins)
    qlf.get_dNdMstar(0.3)
    qlf.get_SMBM(dM, 10.3)
    qlf.get_dNdMbh()
    qlf.get_dNdL([.8,.4], obscured)
    mu = qlf.plotpurp[:,0]
    mulist.append(mu)
    cut.append(np.argmin(qlf.early))

cutlist = []
for i, mul in zip(cut,mulist):
    cutlist.append(mul[i])
    
norm = matplotlib.colors.Normalize(vmin=min(qlf.StellBins), vmax=max(qlf.StellBins))
cb = matplotlib.colorbar.ColorbarBase(cbax, cmap='inferno', norm=norm, orientation='horizontal')
cb.set_label(r'$log_{10}[M_{*}]$')

i_mulist = np.transpose(mulist)
for i, j in zip(i_mulist,qlf.StellBins):
    ax.plot(zlist, i, c = cm.inferno(norm(j)),linewidth = 0.5)

ax.set_ylabel(r'$log_{10} [<\dot{M}_{BH}>/M_{\odot}] yr^{-1}$')
ax.set_xlabel(r'z')
ax.plot(zlist,cutlist,c='k',label='critical mass',linestyle='dotted')
ax.legend()
fig.suptitle(r'Redshift evolution of <$\dot{M}_{BH}$> for varrying $M_{*}$')

fig.show()
#fig.savefig('plots/Mdot_z_evolution.pdf')


FigureCanvasNbAgg()

/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:231: RuntimeWarning: divide by zero encountered in log10
/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:233: RuntimeWarning: divide by zero encountered in log10


In [257]:
#zlist = [0.0, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]
zlist = [0.0,1.0,2.0,3.0,4.0,5.0,6.0]
x = np.linspace(-10,3,500)
obscured = .75
dM = .5
xsig1 = .8
xsig2 = .4
plt.close('all')
count = 1
bigsumg1 = np.zeros(500)
bigsumg2 = np.zeros(500)
bins = 200
for z, c in zip(zlist,['red','gold','mediumspringgreen','deepskyblue','navy','darkorchid','palevioletred','lightcoral','olive','k','gray','aqua','pink','saddlebrown','lime']):
    qlf = QLF(z, bins)
    qlf.get_dNdMstar(0.3)
    qlf.get_SMBM(dM, 10.3)
    qlf.get_dNdMbh()
    qlf.get_dNdL([.8,.4], obscured)
    mu = qlf.plotpurp[:,0]
    xsig = qlf.plotpurp[:,1]
    s1count = 0
    s2count = 0
    sumg1 = np.zeros(500)
    sumg2 = np.zeros(500)
    for m, s in zip(mu,xsig):
        if s == xsig1:
            sumg1 += qlf.gauss(x, m, s, 1)
            s1count += 1
        elif s == xsig2:
            sumg2 += qlf.gauss(x, m, s, 1)
            s2count += 1
    bigsumg1 += sumg1/s1count
    bigsumg2 += sumg2/s2count
#     if count == 1:
#         plt.plot(x,sumg1/s1count,linestyle = 'dashed',color='k', label = 'pre crit mass')
#         plt.plot(x,sumg2/s2count,linestyle = 'solid', label = 'post crit mass',color='k')
#         count+=1
    plt.plot(x,sumg1/s1count,linestyle = 'dashed',color=c,linewidth=1)
    plt.plot(x,sumg2/s2count,linestyle = 'solid', label = 'z = '+str(z),color=c,linewidth=1)
plt.plot(x,bigsumg1/3,linestyle = 'dashed', color = 'k', label = 'pre crit mass')
plt.plot(x,bigsumg2/3,linestyle='solid',c='k', label = 'post crit mass')
    
plt.legend(fontsize=8)
plt.xlabel(r'$log_{10} [<\dot{M}_{BH}>/M_{\odot}] yr^{-1}$')
plt.ylabel(r'probability density')
plt.title(r'$<\dot{M}_{BH}>$ distributions')
plt.savefig('plots/Mdot_distributions.pdf')


FigureCanvasNbAgg()

Code to create universal $<\dot{M}_{BH}>$ model.

In [414]:
import numpy as np
from colossus.cosmology import cosmology
cosmology.setCosmology('planck15') 
from colossus.lss import mass_function as mf 
import glob
import numexpr as ne
import sys
from scipy.optimize import newton
%matplotlib widget


files = [f.split('a')[1].split('.d')[0] for f in glob.glob('ssfrs/ssfr_a*.dat')]
a_list = np.array([float(a) for a in files])
mass_list = np.array([np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,0] for f in files])
ssfr_list = [np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,1] for f in files]
    
param_file = np.loadtxt("smhm_params.txt")
names = "EFF_0 EFF_0_A EFF_0_A2 EFF_0_Z M_1 M_1_A M_1_A2 M_1_Z ALPHA ALPHA_A ALPHA_A2 ALPHA_Z BETA BETA_A BETA_Z DELTA GAMMA GAMMA_A GAMMA_Z CHI2".split(" ");
params = dict(zip(names, param_file[:,1]))

def create_ranges_numexpr(start, stop, N):

    divisor = N-1
    s0 = start[:,None]
    s1 = stop[:,None]
    r = np.arange(N)

    return ne.evaluate('((1.0/divisor) * (s1 - s0))*r + s0')

def grab_obs(redshift):
    
    obs_points = []
    with open("obs_collect.dat", 'r') as f:
        for line in f:
            if line[0:4] != ';;//':
                obs_points.append(line.split())
    x = []
    y = []
    yerr = []
    for i in obs_points:
        if float(i[0]) == redshift:
            x.append(float(i[1]))
            y.append(float(i[2]))
            yerr.append(float(i[3]))
            
    return(x,y,yerr)



class QLF():
    def __init__(self, z, bin_num):
        
        
        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bin_num = bin_num
        self.get_zparams()
        
        self.max_halo = 15.
        self.HaloBins = np.linspace(7., self.max_halo, bin_num)
        slopes = self.get_slope(self.HaloBins)
        while slopes[-1] < 0:
            self.max_halo -= .1
            self.HaloBins = np.linspace(7., self.max_halo, bin_num)
            slopes = self.get_slope(self.HaloBins)
            
        self.max_stell = self.get_Mstar(self.max_halo)
        
        self.fp = self.HaloBins
        self.xp = self.get_Mstar(self.fp)
        
        self.LumBins = np.linspace(5., 16., bin_num)
        self.StellBins = np.linspace(5.,self.max_stell, bin_num)
        
        
    def get_zparams(self):
        a1 = self.a - 1.0
        lna = np.log(self.a)
        self.zparams = {}
        self.zparams['m_1'] = params['M_1'] + a1*params['M_1_A'] - lna*params['M_1_A2'] + self.z*params['M_1_Z']
        self.zparams['sm_0'] = self.zparams['m_1'] + params['EFF_0'] + a1*params['EFF_0_A'] - lna*params['EFF_0_A2'] + self.z*params['EFF_0_Z']
        self.zparams['alpha'] = params['ALPHA'] + a1*params['ALPHA_A'] - lna*params['ALPHA_A2'] + self.z*params['ALPHA_Z']
        self.zparams['beta'] = params['BETA'] + a1*params['BETA_A'] + self.z*params['BETA_Z']
        self.zparams['delta'] = params['DELTA']
        self.zparams['gamma'] = 10**(params['GAMMA'] + a1*params['GAMMA_A'] + self.z*params['GAMMA_Z'])
        
    
    def get_slope(self, Mhalo):

        dm = Mhalo-self.zparams['m_1'];
        term1 = (self.zparams['alpha']*10.**(self.zparams['beta']*dm)+self.zparams['beta']*10.**(self.zparams['alpha']*dm))/(10.**(self.zparams['beta']*dm) + 10.**(self.zparams['alpha']*dm))
        term2 = -self.zparams['gamma']*dm*np.exp(-(dm/self.zparams['delta'])**2/2.)/self.zparams['delta']**2
        slope = term1 + term2

        return slope
    
    def get_Mstar(self, Mhalo):
    
        dm = Mhalo-self.zparams['m_1']
        dm2 = dm/self.zparams['delta']
        Mstar = self.zparams['sm_0'] - np.log10(10**(-self.zparams['alpha']*dm) + 10**(-self.zparams['beta']*dm)) + self.zparams['gamma']*np.exp(-0.5*(dm2*dm2))

        return Mstar
    
    
    def get_Mhalo(self, Mstar):
        
        Mhalo = np.interp(Mstar, self.xp, self.fp)
        
        return Mhalo
    
    
    def get_SMBM(self, dM, Mmid, slope1 = 0.2, slope3 = 1.):

        start = [7., np.log10(1.4*10**4.)]
        stop = [12., np.log10(1.4*10**9.)]
        mstar1 = Mmid - dM
        mstar2 = Mmid + dM
        int1 = start[1] - start[0] * slope1
        int3 = stop[1] - stop[0] * slope3
        x = (int3 - int1) / (slope1 - slope3)
        y = slope1 * x + int1
        if mstar1 < x:
            mstar1 = x
        mbh1 = slope1 * mstar1 + int1
        mbh2 = mstar2 + int3
        slope2 = (mbh2 - mbh1) / (mstar2 - mstar1)
        int2 = mbh2 - mstar2 * slope2

        self.slope_list, self.int_list, self.mass_cuts = [slope1, slope2, slope3], [int1, int2, int3], [mstar1, mstar2] 
    
    
    def gauss_array(self, vals, std, amp):

        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(vals[-1]-vals[:-1])**2.0)/(2.0*std**2))

        return y

    
    def convolve_smhm(self, StellBins, sigma, bin_num, z): 

        halomasses = self.get_Mhalo(np.asarray(StellBins))
        plus_mins = (5.0 * sigma) / self.get_slope(np.asarray(halomasses))
        mins = halomasses - plus_mins
        maxs = halomasses + plus_mins
        mins[mins < 7.] = 7.
        maxs[maxs > self.max_halo] = self.max_halo
        MHalo = create_ranges_numexpr(mins, maxs, bin_num)
        dNdMhalo = mf.massFunction(10.**MHalo, z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16') * np.log(10)
        meanMstar = np.apply_along_axis(self.get_Mstar, 1, MHalo)
        values = np.zeros((bin_num,bin_num+1))
        values[:,-1] = StellBins
        values[:,:-1] = meanMstar
        Mstar_prob = np.apply_along_axis(self.gauss_array, 1, values, sigma, 1)
        dNdMstar = np.sum(Mstar_prob * dNdMhalo, axis = 1) * (MHalo[:,1] - MHalo[:,0])

        return dNdMstar
    
    
    
    def get_dNdMstar(self, smhm_scat):
        
        if smhm_scat == 0.:
            self.dNdMstar = mf.massFunction(10.**self.get_Mhalo(self.StellBins), self.z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')  / (get_slope(um.get_Mhalo(self.StellBins, self.z)) * np.log10(np.e))
        else:
            self.dNdMstar = self.convolve_smhm(self.StellBins, smhm_scat, self.bin_num, self.z)


            
    def get_dNdMbh(self):
        
        self.early = (self.StellBins <= self.mass_cuts[0])
        self.growth = ((self.StellBins > self.mass_cuts[0]) & (self.StellBins < self.mass_cuts[1]))
        self.late = (self.StellBins > self.mass_cuts[1])
        self.m = np.zeros(len(self.StellBins))
        self.m[self.early] = self.slope_list[0]
        self.m[self.growth] = self.slope_list[1]
        self.m[self.late] = self.slope_list[2]
        self.dNdMbh = self.dNdMstar / self.m
        
        
    def etas(self, Mbh):
    
        n = np.asarray(self.LumBins) - np.log10(3.3e4) - Mbh

        return n
        
        
        
            
#     def get_mean_etas(self, vals, a, xsigs, files = files):

#         Mbh = vals[0]
#         Mstar = vals[1]
#         slope = vals[2]
#         if slope == self.slope_list[0]:
#             xsig = xsigs[0]
#         else:
#             xsig = xsigs[1]
#         closest_a = np.argmin(np.abs(a_list - a))
#         closest_m = np.argmin(np.abs(mass_list[closest_a] - Mstar))
#         ssfr = ssfr_list[closest_a][closest_m]

#         Ledd = 1.3*10**31 * 10**Mbh #J/s 
#         Mdotedd = Ledd / (.1 * (2.99*10**8)**2) #kg/s???
#         sbhr = slope*(ssfr/(3.154*10**7)) #1/s??
#         Mdotbh = sbhr*(10**Mbh*2*10**30)
#         eta = Mdotbh/Mdotedd
#         nsig = xsig * np.log10(eta)
        
#         return np.log10(eta), nsig
    
    def get_mean_etas(self, vals, a, xsigs, files = files):

        Mbh = vals[0]
        Mstar = vals[1]
        slope = vals[2]
        if slope == self.slope_list[0]:
            xsig = xsigs[0]
        else:
            xsig = xsigs[1]
        closest_a = np.argmin(np.abs(a_list - a))
        closest_m = np.argmin(np.abs(mass_list[closest_a] - Mstar))
        ssfr = ssfr_list[closest_a][closest_m]

        Ledd = 1.3*10**31 * 10**Mbh #J/s 
        Mdotedd = Ledd / (.1 * (2.99*10**8)**2) #kg/s???
        sbhr = slope*(ssfr/(3.154*10**7)) #1/s??
        Mdotbh = sbhr*(10**Mbh*2*10**30)
        eta = Mdotbh/Mdotedd
        nsig = xsig * np.log10(eta)
        
        return np.log10(eta), nsig, Mdotbh, Mdotedd
    #np.log10(Mdotbh/((2*10**30)*3.17098e-8))
    
    def gauss(self, x, *var):
  
        mean, std, amp = var
        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(x-mean)**2.0)/(2.0*std**2))

        return y
    
    
    def prob_eddratios(self, vals):

        probdens = self.gauss(vals[:-2], vals[-2]-(vals[-1]**2)/2, vals[-1], 1)

        return probdens
    
    def get_dNdL(self, xsigs, obscured):
        
        ###exp sig growth
            
        b = np.zeros(self.bin_num)
        b[self.early] = self.int_list[0]
        b[self.growth] = self.int_list[1]
        b[self.late] = self.int_list[2]

        leftc = np.argmin(self.early)
        rightc = np.argmax(self.late)
        per = len(self.growth[self.growth==True])*.1
        cut1l = int(leftc - per)
        cut1r = int(leftc + per + 1)
        cut2l = int(rightc - per)
        cut2r = int(rightc + per + 1)

        BHBins = self.StellBins * self.m + b
        eta_lists = np.apply_along_axis(self.etas, 1, np.reshape(BHBins,(self.bin_num,1)))
        
        self.eta_lists = eta_lists
        
        vals = np.zeros((self.bin_num,3))
        vals[:,0] = BHBins
        vals[:,1] = self.StellBins
        vals[:,2] = self.m
        mean_etas = np.apply_along_axis(self.get_mean_etas, 1, vals, self.a, xsigs)
            
        vals = np.zeros((self.bin_num, self.bin_num+2))
        vals[:,:-2] = eta_lists
        vals[:,-2] = mean_etas[:,0]
        vals[:,-1] = mean_etas[:,1]
        
        
        self.plotpurp = mean_etas
        
        self.dNdL = np.log10((np.sum(np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1)), axis = 0)))
        ind_dNdL_off = (1-obscured) * np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1))
        self.ind_dNdL = np.zeros((self.bin_num,self.bin_num))
        c = 0
        for l in ind_dNdL_off:
            self.ind_dNdL[c,:] = l
            c += 1


In [455]:
bins = 100
z = 2.0
dM = .5
obscured = .75

fig, ax = plt.subplots(figsize=(6,6))
hist_listpre = []
hist_listpost = []
meanmdotpre = []
meanmdotpost = []

for z in [0.0, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]:
    qlf = QLF(z, bins)
    qlf.get_dNdMstar(0.3)
    qlf.get_SMBM(dM, 10.3)
    qlf.get_dNdMbh()
    qlf.get_dNdL([.8,.4], obscured)
    mean_Mdotbh = qlf.plotpurp[:,2]
    Mdotedd = qlf.plotpurp[:,3]
    etas = qlf.eta_lists


    for n, edd, mean, m in zip(etas, Mdotedd, mean_Mdotbh, qlf.m):
        mdotbh = 10**n * edd
        X = np.log10(mdotbh)/np.log10(mean)
        if m == qlf.slope_list[0]:
            hist_listpre.extend(X)
            meanmdotpre.append(mean)
        else:
            hist_listpost.extend(X)
            meanmdotpost.append(mean)

b = 60
den = True
premean = np.log10(np.mean(meanmdotpre)/(2*10**30*3.17098e-8))
postmean = np.log10(np.mean(meanmdotpost)/(2*10**30*3.17098e-8))

#print(np.log10(np.mean(meanmdotpre)/(2*10**30*3.17098e-8)), np.log10(np.mean(meanmdotpost)/(2*10**30*3.17098e-8)))
# ax.hist(np.array(hist_listpre), density= den, bins = b,color='r', histtype='step', label='pre-disk')
# ax.hist(np.array(hist_listpost), density = den, bins = b, color = 'b', histtype='step',label='post-disk')
# ax.set_xlabel(r'$\frac{log_{10}[\dot{M}_{BH}]}{log_{10} [<\dot{M}_{BH}>]}$')
# ax.legend()
# fig.show()

s1 = 1.25
s2 = .85
x1 = np.linspace(-12,10,500)
# ypre = qlf.gauss(x1, premean, .85, 1)
# ypost = qlf.gauss(x1, postmean, .55, 1)
ypre = qlf.gauss(x1, 1, s1, 1)
ypost = qlf.gauss(x1, 1, s2, 1)

# ax.plot(x1, ypre, c = 'k', label= r'pre-disk: $\log_{10} [<\dot{M}_{BH}>] yr^{-1}$')
# ax.plot(x1, ypost, c= 'r', label=r' post-disk: $\log_{10} [<\dot{M}_{BH}>] yr^{-1}$')
# ax.plot(x1/premean, ypre, c= 'k', label=r'pre-disk: $\frac{log_{10}[\dot{M}_{BH}]}{log_{10} [<\dot{M}_{BH}>]}$', linestyle='dashed')
# ax.plot(x1/postmean, ypost, c='r', label=r'post-disk: $\frac{log_{10}[\dot{M}_{BH}]}{log_{10} [<\dot{M}_{BH}>]}$', linestyle='dashed')
# ax.legend(loc='upper center', bbox_to_anchor=(0.3, 1.))
# ax.axvline(1, color = 'gray', linestyle='dotted')
# ax.text(-7,.6,r'$\sigma_{log_{10}[<\dot{M}_{BH}>]} = .85$',color='k')
# ax.text(-7,.5,r'$\sigma_{log_{10}[<\dot{M}_{BH}>]} = .55$',color='r')
# ax.axis([-8,3,0,1])

prestring = str(premean)[0:5]
poststring = str(postmean)[0:5]
ax.plot(x1, ypre, c = 'k', label=r'pre-disk $X$')
ax.plot(x1, ypost, c= 'r', label=r'post-disk $X$')
ax.axis([-12,8,0,0.5])
#ax.plot(x1*premean, ypre/np.abs(premean), c = 'k', label= r'$\log_{10} [<\dot{M}_{BH}>] = $'+prestring, linestyle='dashed')
#ax.plot(x1*postmean, ypost/np.abs(postmean), c = 'r', label=r'$\log_{10} [<\dot{M}_{BH}>] = $'+poststring, linestyle='dashed')

for m, lw in zip([0,-1,-2,-3,-4,-5,-6,-7],[1.5,1.3,1.1,.9,.7,.5,.3,.1]):
    ax.plot(x1, qlf.gauss(x1, m, s1*m, 1), c = 'k', linestyle='dashed',linewidth = lw)
    ax.plot(x1, qlf.gauss(x1, m, s2*m, 1), c = 'r', linestyle='dashed',linewidth = lw)



ax.text(4,.25,r'$\sigma_{X} = $'+str(s1),color='k')
ax.text(4,.35,r'$\sigma_{X} = $'+str(s2),color='r')

ax.set_xlabel(r'$log_{10}[\dot{M}_{BH}] \ \  yr^{-1}$')
ax.legend(loc='upper center', bbox_to_anchor=(0.25, .8))
ax.axvline(1, color = 'gray', linestyle='dotted')
ax.text(1.2,.47,r'$\longleftarrow X = \frac{log_{10}[\dot{M}_{BH}]}{log_{10} [<\dot{M}_{BH}>]}$',color = 'grey',fontsize=12)
ax.set_ylabel('Probability Density')
ax.axvline(-1, color = 'silver', linestyle = 'dotted')
ax.text(-8,0.47,r'$log_{10} [<\dot{M}_{BH}>] = \mu \longrightarrow$')
ax.text(-7.8,0.44,r'$\sigma_{log_{10}[<\dot{M}_{BH}>]} = \mu \ \sigma_{X}$')

# ax.plot(x1, qlf.gauss(x1, premean, .85*premean, 1), c = 'b', linestyle = 'dotted')
# ax.plot(x1, qlf.gauss(x1, postmean, .45*postmean, 1), c = 'b', linestyle='dotted')

#fig.savefig('plots/universaldist_mdot_v2.1.pdf')


/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/matplotlib/pyplot.py:537: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  max_open_warning, RuntimeWarning)


FigureCanvasNbAgg()

/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:239: RuntimeWarning: divide by zero encountered in double_scalars
/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:239: RuntimeWarning: divide by zero encountered in true_divide
/Users/megantillman/anaconda3/envs/astroconda/lib/python3.6/site-packages/ipykernel_launcher.py:239: RuntimeWarning: invalid value encountered in multiply


In [533]:
import numpy as np
from colossus.cosmology import cosmology
cosmology.setCosmology('planck15') 
from colossus.lss import mass_function as mf 
import glob
import numexpr as ne
import sys
from scipy.optimize import newton
%matplotlib widget


files = [f.split('a')[1].split('.d')[0] for f in glob.glob('ssfrs/ssfr_a*.dat')]
a_list = np.array([float(a) for a in files])
mass_list = np.array([np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,0] for f in files])
ssfr_list = [np.loadtxt("ssfrs/ssfr_a"+f+".dat")[:,1] for f in files]
    
param_file = np.loadtxt("smhm_params.txt")
names = "EFF_0 EFF_0_A EFF_0_A2 EFF_0_Z M_1 M_1_A M_1_A2 M_1_Z ALPHA ALPHA_A ALPHA_A2 ALPHA_Z BETA BETA_A BETA_Z DELTA GAMMA GAMMA_A GAMMA_Z CHI2".split(" ");
params = dict(zip(names, param_file[:,1]))

def create_ranges_numexpr(start, stop, N):

    divisor = N-1
    s0 = start[:,None]
    s1 = stop[:,None]
    r = np.arange(N)

    return ne.evaluate('((1.0/divisor) * (s1 - s0))*r + s0')

def grab_obs(redshift):
    
    obs_points = []
    with open("obs_collect.dat", 'r') as f:
        for line in f:
            if line[0:4] != ';;//':
                obs_points.append(line.split())
    x = []
    y = []
    yerr = []
    for i in obs_points:
        if float(i[0]) == redshift:
            x.append(float(i[1]))
            y.append(float(i[2]))
            yerr.append(float(i[3]))
            
    return(x,y,yerr)



class QLF():
    def __init__(self, z, bin_num):
        
        
        self.z = float(z)
        self.a = 1.0/(1.0+self.z)
        self.bin_num = bin_num
        self.get_zparams()
        
        self.max_halo = 15.
        self.HaloBins = np.linspace(7., self.max_halo, bin_num)
        slopes = self.get_slope(self.HaloBins)
        while slopes[-1] < 0:
            self.max_halo -= .1
            self.HaloBins = np.linspace(7., self.max_halo, bin_num)
            slopes = self.get_slope(self.HaloBins)
            
        self.max_stell = self.get_Mstar(self.max_halo)
        
        self.fp = self.HaloBins
        self.xp = self.get_Mstar(self.fp)
        
        self.LumBins = np.linspace(5., 16., bin_num)
        self.StellBins = np.linspace(5.,self.max_stell, bin_num)
        
        
    def get_zparams(self):
        a1 = self.a - 1.0
        lna = np.log(self.a)
        self.zparams = {}
        self.zparams['m_1'] = params['M_1'] + a1*params['M_1_A'] - lna*params['M_1_A2'] + self.z*params['M_1_Z']
        self.zparams['sm_0'] = self.zparams['m_1'] + params['EFF_0'] + a1*params['EFF_0_A'] - lna*params['EFF_0_A2'] + self.z*params['EFF_0_Z']
        self.zparams['alpha'] = params['ALPHA'] + a1*params['ALPHA_A'] - lna*params['ALPHA_A2'] + self.z*params['ALPHA_Z']
        self.zparams['beta'] = params['BETA'] + a1*params['BETA_A'] + self.z*params['BETA_Z']
        self.zparams['delta'] = params['DELTA']
        self.zparams['gamma'] = 10**(params['GAMMA'] + a1*params['GAMMA_A'] + self.z*params['GAMMA_Z'])
        
    
    def get_slope(self, Mhalo):

        dm = Mhalo-self.zparams['m_1'];
        term1 = (self.zparams['alpha']*10.**(self.zparams['beta']*dm)+self.zparams['beta']*10.**(self.zparams['alpha']*dm))/(10.**(self.zparams['beta']*dm) + 10.**(self.zparams['alpha']*dm))
        term2 = -self.zparams['gamma']*dm*np.exp(-(dm/self.zparams['delta'])**2/2.)/self.zparams['delta']**2
        slope = term1 + term2

        return slope
    
    def get_Mstar(self, Mhalo):
    
        dm = Mhalo-self.zparams['m_1']
        dm2 = dm/self.zparams['delta']
        Mstar = self.zparams['sm_0'] - np.log10(10**(-self.zparams['alpha']*dm) + 10**(-self.zparams['beta']*dm)) + self.zparams['gamma']*np.exp(-0.5*(dm2*dm2))

        return Mstar
    
    
    def get_Mhalo(self, Mstar):
        
        Mhalo = np.interp(Mstar, self.xp, self.fp)
        
        return Mhalo
    
    
    def get_SMBM(self, dM, Mmid, slope1 = 0.2, slope3 = 1.):

        start = [7., np.log10(1.4*10**4.)]
        stop = [12., np.log10(1.4*10**9.)]
        mstar1 = Mmid - dM
        mstar2 = Mmid + dM
        int1 = start[1] - start[0] * slope1
        int3 = stop[1] - stop[0] * slope3
        x = (int3 - int1) / (slope1 - slope3)
        y = slope1 * x + int1
        if mstar1 < x:
            mstar1 = x
        mbh1 = slope1 * mstar1 + int1
        mbh2 = mstar2 + int3
        slope2 = (mbh2 - mbh1) / (mstar2 - mstar1)
        int2 = mbh2 - mstar2 * slope2

        self.slope_list, self.int_list, self.mass_cuts = [slope1, slope2, slope3], [int1, int2, int3], [mstar1, mstar2] 
    
    
    def gauss_array(self, vals, std, amp):

        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(vals[-1]-vals[:-1])**2.0)/(2.0*std**2))

        return y

    
    def convolve_smhm(self, StellBins, sigma, bin_num, z): 

        halomasses = self.get_Mhalo(np.asarray(StellBins))
        plus_mins = (5.0 * sigma) / self.get_slope(np.asarray(halomasses))
        mins = halomasses - plus_mins
        maxs = halomasses + plus_mins
        mins[mins < 7.] = 7.
        maxs[maxs > self.max_halo] = self.max_halo
        MHalo = create_ranges_numexpr(mins, maxs, bin_num)
        dNdMhalo = mf.massFunction(10.**MHalo, z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16') * np.log(10)
        meanMstar = np.apply_along_axis(self.get_Mstar, 1, MHalo)
        values = np.zeros((bin_num,bin_num+1))
        values[:,-1] = StellBins
        values[:,:-1] = meanMstar
        Mstar_prob = np.apply_along_axis(self.gauss_array, 1, values, sigma, 1)
        dNdMstar = np.sum(Mstar_prob * dNdMhalo, axis = 1) * (MHalo[:,1] - MHalo[:,0])

        return dNdMstar
    
    
    
    def get_dNdMstar(self, smhm_scat):
        
        if smhm_scat == 0.:
            self.dNdMstar = mf.massFunction(10.**self.get_Mhalo(self.StellBins), self.z, q_in='M', q_out='dndlnM', mdef='vir', model='despali16')  / (get_slope(um.get_Mhalo(self.StellBins, self.z)) * np.log10(np.e))
        else:
            self.dNdMstar = self.convolve_smhm(self.StellBins, smhm_scat, self.bin_num, self.z)


            
    def get_dNdMbh(self):
        
        self.early = (self.StellBins <= self.mass_cuts[0])
        self.growth = ((self.StellBins > self.mass_cuts[0]) & (self.StellBins < self.mass_cuts[1]))
        self.late = (self.StellBins > self.mass_cuts[1])
        self.m = np.zeros(len(self.StellBins))
        self.m[self.early] = self.slope_list[0]
        self.m[self.growth] = self.slope_list[1]
        self.m[self.late] = self.slope_list[2]
        self.dNdMbh = self.dNdMstar / self.m
        
        
    def etas(self, Mbh):
    
        n = np.asarray(self.LumBins) - np.log10(3.3e4) - Mbh

        return n
        
        
        
            
#     def get_mean_etas(self, vals, a, xsigs, files = files):

#         Mbh = vals[0]
#         Mstar = vals[1]
#         slope = vals[2]
#         if slope == self.slope_list[0]:
#             xsig = xsigs[0]
#         else:
#             xsig = xsigs[1]
#         closest_a = np.argmin(np.abs(a_list - a))
#         closest_m = np.argmin(np.abs(mass_list[closest_a] - Mstar))
#         ssfr = ssfr_list[closest_a][closest_m]

#         Ledd = 1.3*10**31 * 10**Mbh #J/s 
#         Mdotedd = Ledd / (.1 * (2.99*10**8)**2) #kg/s???
#         sbhr = slope*(ssfr/(3.154*10**7)) #1/s??
#         Mdotbh = sbhr*(10**Mbh*2*10**30)
#         eta = Mdotbh/Mdotedd
#         nsig = xsig * np.log10(eta)
        
#         return np.log10(eta), nsig
    
    def get_mean_etas(self, vals, a, xsigs, files = files):

        Mbh = vals[0]
        Mstar = vals[1]
        slope = vals[2]
        if slope == self.slope_list[0]:
            xsig = xsigs[0]
        else:
            xsig = xsigs[1]
        closest_a = np.argmin(np.abs(a_list - a))
        masses = np.array(mass_list[closest_a])
        ssfrs = np.array(ssfr_list[closest_a])
        closest_m = np.argmin(np.abs(masses - Mstar))
        nonzero = (ssfrs != 0)
        minm = np.min(masses[nonzero])
        maxm = np.max(masses[nonzero])
        if minm < Mstar < maxm:
            ssfr = np.interp(Mstar, masses[nonzero], ssfrs[nonzero])
        else:
            ssfr = ssfr_list[closest_a][closest_m]
        
        flag = 0
        if Mstar > maxm:
            flag = 1


        Ledd = 1.3*10**31 * 10**Mbh #J/s 
        Mdotedd = Ledd / (.1 * (2.99*10**8)**2) #kg/s???
        sbhr = slope*(ssfr/(3.154*10**7)) #1/s??
        Mdotbh = sbhr*(10**Mbh*2*10**30)
        eta = Mdotbh/Mdotedd
        nsig = xsig * np.log10(eta)
        
        return np.log10(eta), nsig, Mdotbh, Mdotedd, Mstar, ssfr, flag, Mbh, sbhr
    
    def gauss(self, x, *var):
  
        mean, std, amp = var
        y = (amp/np.sqrt(2.0*np.pi*std**2.0))*np.exp((-(x-mean)**2.0)/(2.0*std**2))

        return y
    
    
    def prob_eddratios(self, vals):

        probdens = self.gauss(vals[:-2], vals[-2]-(vals[-1]**2)/2, vals[-1], 1)

        return probdens
    
    def get_dNdL(self, xsigs, obscured):
        
        ###exp sig growth
            
        b = np.zeros(self.bin_num)
        b[self.early] = self.int_list[0]
        b[self.growth] = self.int_list[1]
        b[self.late] = self.int_list[2]

        leftc = np.argmin(self.early)
        rightc = np.argmax(self.late)
        per = len(self.growth[self.growth==True])*.1
        cut1l = int(leftc - per)
        cut1r = int(leftc + per + 1)
        cut2l = int(rightc - per)
        cut2r = int(rightc + per + 1)

        BHBins = self.StellBins * self.m + b
        eta_lists = np.apply_along_axis(self.etas, 1, np.reshape(BHBins,(self.bin_num,1)))
        
        self.eta_lists = eta_lists
        
        vals = np.zeros((self.bin_num,3))
        vals[:,0] = BHBins
        vals[:,1] = self.StellBins
        vals[:,2] = self.m
        mean_etas = np.apply_along_axis(self.get_mean_etas, 1, vals, self.a, xsigs)
            
        vals = np.zeros((self.bin_num, self.bin_num+2))
        vals[:,:-2] = eta_lists
        vals[:,-2] = mean_etas[:,0]
        vals[:,-1] = mean_etas[:,1]
        
        
        self.plotpurp = mean_etas
        
        self.dNdL = np.log10((np.sum(np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1)), axis = 0)))
        ind_dNdL_off = (1-obscured) * np.apply_along_axis(self.prob_eddratios, 1, vals) * np.reshape(self.dNdMbh,(self.bin_num,1)) * (self.StellBins[1] - self.StellBins[0]) * np.reshape(self.m,(self.bin_num,1))
        self.ind_dNdL = np.zeros((self.bin_num,self.bin_num))
        c = 0
        for l in ind_dNdL_off:
            self.ind_dNdL[c,:] = l
            c += 1


In [551]:
plt.close('all')
bins = 50
fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(1,5,figsize=(30,5))
obscured = .8
sig_X = [1.25,.85]
dM = .55
mMid = 10.3
smhm_scat = 0.3
for z, c in zip([0.5,1.0,1.5,2.0,2.5,3.0,4.0],['r','orange','gold','green','blue','violet','indigo','']):
    qlf = QLF(z, bins)
    qlf.get_dNdMstar(smhm_scat)
    qlf.get_SMBM(dM, mMid)
    qlf.get_dNdMbh()
    qlf.get_dNdL(sig_X, obscured)

    logMstar = qlf.plotpurp[:,4]
    ssfr = qlf.plotpurp[:,5] #yr^-1
    sfr = ssfr * 10**logMstar
    Mdotbh = qlf.plotpurp[:,2]/(2*10**30*3.17098e-8)
    flags = qlf.plotpurp[:,6]
    logMbh = qlf.plotpurp[:,7]
    sbhr = qlf.plotpurp[:,8]
    lw = 1. 
    
    ax1.plot(logMstar, np.log10(Mdotbh),label='z = '+str(z),color = c,linewidth = lw)
    ax1.set_xlabel(r'$log_{10}[M_{*}/M_{\odot}]$')
    ax1.set_ylabel(r'$log_{10}[<\dot{M}_{BH}>] \ \ yr^{-1}$')
    
    ax2.plot(np.log10(sfr), np.log10(Mdotbh),label='z = '+str(z),color = c,linewidth = lw)
    ax2.set_xlabel(r'$log_{10}[<\dot{M}_{*}>] \ \ yr^{-1}$')
    ax2.set_ylabel(r'$log_{10}[<\dot{M}_{BH}>] \ \ yr^{-1}$')
    
    ax3.plot(logMstar, logMbh, color = c, linewidth = lw)
    ax3.set_xlabel(r'$log_{10}[M_{*}/M_{\odot}]$')
    ax3.set_ylabel(r'$log_{10}[M_{BH}/M_{\odot}]$')
    
    ax4.plot(logMstar, logMbh/logMstar, color = c, linewidth = lw)
    ax4.set_xlabel(r'$log_{10}[M_{*}/M_{\odot}]$')    
    ax4.set_ylabel(r'$log_{10}[M_{BH}/M_{*}]$')
    
    ax5.plot(logMstar, np.log10(sfr), color = c, linewidth = lw)
    ax5.set_xlabel(r'$log_{10}[M_{*}/M_{\odot}]$')
    ax5.set_ylabel(r'$log_{10}[<\dot{M}_{*}>] \ \ yr^{-1}$')
    
plt.tight_layout()
ax1.legend()
fig.savefig('plots/MdotBH_Mstar_SFR_relation.pdf')

FigureCanvasNbAgg()